# Calibration: 3-Channel Posterior Reliability
**File:** `calibration_3channel_posterior.ipynb` (renamed from `mvp_calibration.ipynb`)

## Purpose
Evaluates whether the 3-channel Bayesian posterior is calibrated — i.e., when it says P=0.8, is the compound correct ~80% of the time? Generates reliability diagrams stratified by entropy similarity bin.

## What is done
- Loads `out_proposal/assertions.csv` posteriors
- Reliability diagram (overall): ECE computed over confirmed candidates
- Reliability diagram stratified by entropy similarity bin (<0.5, 0.5-0.7, 0.7-0.9, >0.9)
- Exposes where the model fails across difficulty regimes

## Status
SUPERSEDED — Calibration analysis is now fully integrated into `channel_ceiling.ipynb` (Steps 5-7) with bin-specific Platt scaling. This notebook is retained as a simpler standalone reference.

## Key finding
Model is monotonically calibrated overall. High-sim regime (>0.9) is well-calibrated; low-sim regime (<0.5) shows underconfidence. Bin-specific Platt scaling in channel_ceiling.ipynb fixes this.


# MVP Calibration: 3-Channel Posterior Reliability

## What we're doing

Using the full 3-channel posterior (`post`) already computed in `assertions.csv` to evaluate whether the pipeline is calibrated — i.e., when it says P=0.8, is the candidate actually correct 80% of the time?

Reliability diagrams split by entropy similarity bin (<0.5, 0.5–0.7, 0.7–0.9, >0.9) to expose where the model fails across difficulty regimes.

In [ ]:
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings('ignore')

ROOT    = '/Users/ellayoung/Desktop/metabolo_confi_score'
OUT_DIR = f'{ROOT}/out_proposal'

# Load the 3-channel posteriors — already computed
assertions = pd.read_csv(f'{OUT_DIR}/assertions.csv')
confirmed  = assertions[assertions['confirmed']].copy()
confirmed['label'] = confirmed['correct'].astype(int)

print('Confirmed candidates: %d' % len(confirmed))
print('Fraction correct:     %.3f' % confirmed['label'].mean())
print('Posterior stats:')
print(confirmed['post'].describe())


## Reliability Diagrams

Each (spectrum, candidate) row has a 3-channel posterior `post` and a binary label (`correct`).
Bin by predicted posterior → plot mean predicted vs fraction actually correct.

**Evaluation set:** all candidates from confirmed spectra.  
36.6% of candidates are correct (1 correct compound + multiple wrong candidates per spectrum).

In [ ]:
def reliability_diagram(ax, posteriors, labels, n_bins=10, color='steelblue',
                        label='model'):
    bins   = np.linspace(0, 1, n_bins + 1)
    mean_p, frac_c, counts = [], [], []
    for lo, hi in zip(bins[:-1], bins[1:]):
        mask = (posteriors >= lo) & (posteriors < hi)
        if mask.sum() == 0:
            continue
        mean_p.append(posteriors[mask].mean())
        frac_c.append(labels[mask].mean())
        counts.append(mask.sum())
    mean_p = np.array(mean_p)
    frac_c = np.array(frac_c)
    counts = np.array(counts)
    ece = (counts / counts.sum() * np.abs(mean_p - frac_c)).sum()

    ax.plot([0, 1], [0, 1], 'k--', lw=0.8, label='perfect')
    ax.scatter(mean_p, frac_c, s=counts / counts.max() * 300,
               color=color, alpha=0.7, zorder=3,
               label='%s  ECE=%.3f' % (label, ece))
    ax.plot(mean_p, frac_c, color=color, alpha=0.4, lw=1.2)
    ax.set_xlim(0, 1); ax.set_ylim(0, 1)
    ax.set_xlabel('Mean predicted probability')
    ax.set_ylabel('Fraction correct')
    ax.legend(fontsize=8)
    return ece


# ── Overall ────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(6, 5))
ece = reliability_diagram(ax, confirmed['post'].values, confirmed['label'].values,
                          label='3-channel posterior')
ax.set_title('Overall reliability — 3-channel posterior  ECE=%.3f' % ece)
plt.tight_layout()
plt.savefig(f'{ROOT}/viz/mvp_reliability_overall.png', dpi=150)
plt.show()
print('Overall ECE: %.4f' % ece)


In [ ]:
# ── By entropy similarity bin ──────────────────────────────────────────────
sim_bins = [
    (0.0,  0.5,  'sim < 0.5'),
    (0.5,  0.7,  'sim 0.5–0.7'),
    (0.7,  0.9,  'sim 0.7–0.9'),
    (0.9,  1.01, 'sim > 0.9'),
]

fig, axes = plt.subplots(2, 2, figsize=(12, 10))
fig.suptitle('3-Channel Posterior Reliability by Entropy Similarity Bin', fontsize=12)
axes = axes.flatten()

summary = []
for i, (lo, hi, lbl) in enumerate(sim_bins):
    mask = (confirmed['entropy_similarity'] >= lo) & (confirmed['entropy_similarity'] < hi)
    sub  = confirmed[mask]
    ax   = axes[i]

    if len(sub) < 10 or sub['label'].sum() == 0:
        ax.text(0.5, 0.5, 'n=%d\n(too few)' % len(sub),
                ha='center', va='center', transform=ax.transAxes)
        ax.set_title(lbl)
        continue

    ece = reliability_diagram(ax, sub['post'].values, sub['label'].values,
                              label='n=%d' % len(sub))
    ax.set_title('%s  |  ECE=%.3f  |  frac_correct=%.2f' % (lbl, ece, sub['label'].mean()))
    summary.append({'bin': lbl, 'n': len(sub), 'frac_correct': sub['label'].mean(), 'ECE': ece})

plt.tight_layout()
plt.savefig(f'{ROOT}/viz/mvp_reliability_by_sim_bin.png', dpi=150)
plt.show()

print('\n%-14s  %8s  %10s  %6s' % ('bin', 'n_cand', 'frac_corr', 'ECE'))
print('-' * 45)
for r in summary:
    print('%-14s  %8d  %10.3f  %6.4f' % (r['bin'], r['n'], r['frac_correct'], r['ECE']))
